# 03 — NASA CMAPSS Exploration

**Dataset:** NASA CMAPSS — Turbofan Engine Degradation Simulation  
**Goal:** Understand the data structure, sensor distributions, RUL behaviour, and define failure labels for classification.

## Download
Place the files in `data/raw/cmapss/`:
```
train_FD001.txt  test_FD001.txt  RUL_FD001.txt  (repeat for FD002-FD004)
```
Source: https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.cmapss_loader import CMAPSSLoader, CMAPSS_COLUMNS, CONSTANT_SENSORS
from src.data.preprocessor import TimeSeriesPreprocessor

sns.set_theme(style='darkgrid')
pd.set_option('display.max_columns', 30)
print('Imports OK')

## 1. Load FD001

In [ ]:
loader = CMAPSSLoader(data_dir='../data/raw/cmapss')
train_df, test_df = loader.load(subset='FD001')

print(f'Train shape : {train_df.shape}')
print(f'Test shape  : {test_df.shape}')
train_df.head()

In [ ]:
sensor_cols = [c for c in CMAPSS_COLUMNS if c.startswith('sensor')]
train_df[sensor_cols].describe().T

## 2. Units and Cycle Lengths

In [ ]:
unit_cycles = train_df.groupby('unit_id')['cycle'].max().sort_values()
print(f'Units: {len(unit_cycles)}  |  Min lifetime: {unit_cycles.min()}  |  Max: {unit_cycles.max()}  |  Mean: {unit_cycles.mean():.1f}')

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(unit_cycles.index, unit_cycles.values, width=0.8)
ax.set(xlabel='Unit ID', ylabel='Total cycles', title='FD001 — Engine lifetime per unit')
plt.tight_layout(); plt.show()

## 3. RUL Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_df['RUL'], bins=50, color='steelblue', edgecolor='white')
axes[0].set(xlabel='RUL (cycles)', ylabel='Count', title='Train RUL distribution')

for uid in train_df['unit_id'].unique()[:5]:
    sub = train_df[train_df['unit_id'] == uid]
    axes[1].plot(sub['cycle'], sub['RUL'], label=f'Unit {uid}')
axes[1].set(xlabel='Cycle', ylabel='RUL', title='RUL over time (5 sample units)')
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## 4. Failure Label (RUL threshold = 30 cycles)

In [ ]:
train_df = loader.add_failure_label(train_df, rul_threshold=30)

lc = train_df['failure_imminent'].value_counts()
print(f'Normal (0): {lc[0]:,} rows ({lc[0]/len(train_df)*100:.1f}%)')
print(f'Failure (1): {lc[1]:,} rows ({lc[1]/len(train_df)*100:.1f}%)')

unit1 = train_df[train_df['unit_id'] == 1]
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(unit1['cycle'], unit1['RUL'], color='steelblue', label='RUL')
ax.axhline(30, color='red', linestyle='--', label='Threshold (30)')
ax.fill_between(unit1['cycle'], 0, unit1['RUL'], where=unit1['failure_imminent']==1,
                alpha=0.3, color='red', label='Failure zone')
ax.set(xlabel='Cycle', ylabel='RUL', title='Unit 1 — failure zone highlighted')
ax.legend(); plt.tight_layout(); plt.show()

## 5. Constant Sensors (to drop)

In [ ]:
print('Sensor std devs (ascending):')
print(train_df[sensor_cols].std().sort_values().to_string())
print(f'\nDropping: {CONSTANT_SENSORS}')

train_clean = loader.drop_constant_sensors(train_df)
test_clean  = loader.drop_constant_sensors(test_df)
active_sensors = [c for c in sensor_cols if c not in CONSTANT_SENSORS]
print(f'Active sensors ({len(active_sensors)}): {active_sensors}')

## 6. Sensor Correlation Heatmap

In [ ]:
corr = train_clean[active_sensors].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, linewidths=0.5, ax=ax)
ax.set_title('Sensor correlation matrix (FD001 train)')
plt.tight_layout(); plt.show()

## 7. Sensor Trends Over Lifetime (Unit 1)

In [ ]:
unit1_clean = train_clean[train_clean['unit_id'] == 1]
plot_sensors = active_sensors[:9]

fig, axes = plt.subplots(3, 3, figsize=(14, 9), sharex=True)
for ax, col in zip(axes.flat, plot_sensors):
    ax.plot(unit1_clean['cycle'], unit1_clean[col], linewidth=0.8)
    ax.axvline(unit1_clean['cycle'].max() - 30, color='red', linestyle='--', linewidth=0.8)
    ax.set_title(col, fontsize=9)
fig.suptitle('Unit 1 — sensor readings (red line = failure zone start)', fontsize=12)
plt.tight_layout(); plt.show()

## 8. Normalise and Create Windows

In [ ]:
prep = TimeSeriesPreprocessor()
train_norm = prep.normalise(train_clean, method='minmax', feature_cols=active_sensors)
test_norm  = prep.transform(test_clean, feature_cols=active_sensors)
test_norm  = loader.add_failure_label(test_norm, rul_threshold=30)

WINDOW = 30
X_list, y_list = [], []
for uid in train_norm['unit_id'].unique():
    unit_df = train_norm[train_norm['unit_id'] == uid]
    X_u, y_u = prep.create_windows(unit_df, active_sensors, 'failure_imminent',
                                   window_size=WINDOW, step=1)
    X_list.append(X_u); y_list.append(y_u)

X = np.concatenate(X_list)
y = np.concatenate(y_list)
print(f'X shape : {X.shape}  -> (samples, window, features)')
print(f'y shape : {y.shape}')
print(f'Class balance — Normal: {(y==0).sum():,}  |  Failure: {(y==1).sum():,}')

## Summary

| Item | Value |
|------|-------|
| Subset | FD001 (1 operating condition, 1 fault mode) |
| Training engines | 100 |
| Active sensors | 15 (6 constant sensors dropped) |
| Window size | 30 cycles |
| Failure threshold | RUL ≤ 30 cycles |
| Final X shape | (n_windows, 30, 15) |

**Next:** feed `X`, `y` into `LSTMFailureClassifier` in `04_shap_prototyping.ipynb`.